In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.profiler import profile, record_function, ProfilerActivity
from transformers import Trainer, TrainingArguments, TrainerCallback

# 1. Prepare the CIFAR-10 dataset wrapped in a custom Dataset class.
class CIFAR10Dataset(torch.utils.data.Dataset):
    def __init__(self, split: str, transform):
        # If split=="train", use the training split; else use the test split.
        self.data = torchvision.datasets.CIFAR10(
            root='./data',
            train=(split == 'train'),
            download=True,
            transform=transform
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        image, label = self.data[idx]
        # Return a dict so that Trainer can unpack it into model(**inputs)
        return {"pixel_values": image, "labels": label}

# 2. Define image transforms.
transform = transforms.Compose([
    # Resize CIFAR10 images (32x32) to 224x224 to work with standard ResNet.
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Create training and evaluation datasets.
train_dataset = CIFAR10Dataset(split="train", transform=transform)
eval_dataset = CIFAR10Dataset(split="test", transform=transform)

# 3. Define the ResNet model wrapped in a custom nn.Module.
class CIFAR10ResNet(nn.Module):
    def __init__(self):
        super(CIFAR10ResNet, self).__init__()
        # Use a ResNet-18 architecture from torchvision.
        self.resnet = torchvision.models.resnet18(pretrained=False)
        # Replace the final fully connected layer to output 10 classes.
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 10)

    def forward(self, pixel_values, labels=None):
        logits = self.resnet(pixel_values)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        # When loss is computed, return it along with logits (Trainer expects a dict)
        if loss is not None:
            return {"loss": loss, "logits": logits}
        else:
            return logits

# Instantiate the model.
model = CIFAR10ResNet()

# 4. Set up the Hugging Face Trainer arguments.
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
    report_to="tensorboard"  # so that logs (if any) are sent to TensorBoard
)

# 5. Define a custom TrainerCallback that wraps the PyTorch profiler.
class TorchMemoryProfilerCallback(TrainerCallback):
    def __init__(self):
        # Define profiler activities: include CUDA if available.
        activities = [ProfilerActivity.CPU]
        if torch.cuda.is_available():
            activities.append(ProfilerActivity.CUDA)
        # Create a profiler instance with a schedule.
        self.profiler = profile(
            activities=activities,
            schedule=torch.profiler.schedule(
                wait=1,  # number of steps to wait before warming up
                warmup=1,  # number of warmup steps
                active=3,  # number of steps to record per cycle
                repeat=1  # number of cycles
            ),
            on_trace_ready=torch.profiler.tensorboard_trace_handler("./log/memory"),
            record_shapes=True,
            profile_memory=True,
            with_stack=True
        )

    def on_train_begin(self, args, state, control, **kwargs):
        print("Starting profiler...")
        self.profiler.start()

    def on_step_end(self, args, state, control, **kwargs):
        # Step the profiler after every training step.
        self.profiler.step()

    def on_train_end(self, args, state, control, **kwargs):
        print("Stopping profiler...")
        self.profiler.stop()

# 6. Instantiate the Trainer with our model, datasets, and the profiler callback.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[TorchMemoryProfilerCallback()]
)

# 7. Start training.
trainer.train()

# After training, to visualize the profiling results (including memory usage), run:
#   tensorboard --logdir=./log/memory
# Then open the provided URL in your web browser.
